# Clase 04 — Seguimiento de experimentos

**Curso:** MLOps y Monitorización con Python · **Bloque 2:** Versionar y medir
**Tiempo estimado:** 105 minutos · **Capítulo del manual:** `manual/04_seguimiento_experimentos.md`

## Qué vas a ver en esta clase

- Por qué anotar experimentos "a mano" (en un cuaderno o en el nombre del archivo) deja de funcionar a partir de un puñado de pruebas.
- **MLflow Tracking** con un almacén de archivos **local** (`mlflow.set_tracking_uri("file:" + ...)`, nunca un servidor), dentro de la carpeta de trabajo efímera.
- Registrar de verdad, con `mlflow.start_run()`, `log_param`, `log_metric` y `log_artifact`, **10 configuraciones reales** de un modelo sobre `datos/churn.csv`.
- Leer los runs guardados con `mlflow.search_runs()` (un `DataFrame` de pandas) y compararlos con un **criterio explícito**: no solo la métrica principal, también un indicador de coste y la estabilidad entre semillas.
- Registrar el modelo elegido como **artefacto versionado** con `mlflow.sklearn.log_model`, primer paso hacia el registro de modelos de la clase 8.

Todo ocurre dentro de una carpeta de trabajo efímera (`proyecto_churn/`), creada al principio y **eliminada al final** de este mismo notebook.

## Requisitos

- `pandas`, `scikit-learn` y `mlflow` (la celda siguiente instala solo lo que falte, también en Google Colab).
- La carpeta `datos/` del curso, con `churn.csv` dentro, y haber leído `datos/LEEME.md` (proporción de abandono, exactitud engañosa).
- De la clase 3: la idea de carpeta de trabajo efímera y de "remoto"/almacén local.

> **Cómo usar este notebook:** ejecuta las celdas en orden, de arriba a abajo, una sola vez. Tarda menos de un minuto en total.

---
## Instalación

Instala solo lo que falte.

In [ ]:
import os
import subprocess
import sys

# Fijamos estas variables ANTES de la primera vez que se importe mlflow (incluso la comprobación de abajo),
# para que ningún mensaje automático de MLflow dependa de la hora del sistema en la salida de este notebook
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"


def asegurar(paquete, modulo):
    """Instala el paquete solo si falta (idempotente y silencioso)."""
    try:
        __import__(modulo)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paquete], check=True)


asegurar("numpy", "numpy")
asegurar("pandas", "pandas")
asegurar("scikit-learn", "sklearn")
asegurar("mlflow", "mlflow")
print("Librerías listas.")

---
## Celda 0 — Setup

Importamos las librerías, definimos `cargar_datos` y preparamos las variables de entorno de MLflow. Todos los entrenamientos de esta clase usan `SEMILLA_BASE = 42` (fijada más abajo), así que no hace falta ninguna otra semilla adicional salvo cuando comparamos explícitamente varias semillas (sección 7).

In [ ]:
import os
import shutil
import time

import sklearn
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, classification_report

def cargar_datos(nombre):
    """Busca un archivo de datos en datos/, ../datos/ y /content/datos/."""
    for base in ("datos", "../datos", "/content/datos"):
        ruta = os.path.join(base, nombre)
        if os.path.exists(ruta):
            return ruta
    # Copia pubblica (Colab): se non c'è in locale, scarica il file dal repository dei notebook.
    try:
        import urllib.request
        destino = os.path.join("datos", nombre)
        os.makedirs(os.path.dirname(destino), exist_ok=True)
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/RaffaeleBini/arcadia-notebooks/main/18-mlops-monitoraggio/datos/{nombre}", destino)
        return destino
    except Exception:
        pass
    raise FileNotFoundError(
        "No encuentro " + nombre + ". Sube la carpeta datos/ del curso (en Colab: panel de archivos) o déjala junto al notebook.")

# Variables de entorno de MLflow: quitan el aviso del asistente, permiten el almacén de archivos
# (en MLflow 3.x el backend de archivos está en modo mantenimiento y hay que aceptarlo explícitamente)
# y desactivan la barra de progreso al guardar/cargar el modelo (no aporta nada en un script y no es reproducible en texto)
os.environ["MLFLOW_DISABLE_AGENT_HINT"] = "1"
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
os.environ["MLFLOW_ENABLE_ARTIFACTS_PROGRESS_BAR"] = "false"

import mlflow
import mlflow.sklearn
import logging

# Los mensajes INFO del propio MLflow incluyen la hora del sistema: los silenciamos para que
# la salida de este notebook no dependa del reloj de tu ordenador
logging.getLogger("mlflow").setLevel(logging.ERROR)  # WARNING no basta: mlflow.utils.requirements_utils avisa en WARNING con detalles no deterministicos

print("Librerías y funciones auxiliares listas. Versión de MLflow:", mlflow.__version__)

---
## 1. Por qué un cuaderno de notas deja de funcionar

Simulamos, con solo 3 líneas, cómo se anotan a mano los experimentos en muchos proyectos reales al principio.

In [ ]:
# Así es como se anotan los experimentos "a mano" en un cuaderno o en el nombre del archivo
cuaderno_manual = [
    "modelo_v1.pkl -- regresion logistica, sin balancear, acc como 0.88 creo",
    "modelo_v2_bueno.pkl -- probé class_weight balanced, mejor pero no anoté el AUC",
    "modelo_v2_final_DEFINITIVO.pkl -- otra vez balanced, cambié algo de C, no recuerdo qué valor",
]
for linea in cuaderno_manual:
    print("-", linea)

# Pregunta concreta: ¿cuál de los tres tiene mejor AUC? ¿con qué C se entrenó el segundo?
# Respuesta con este "cuaderno": no se puede saber sin adivinar o sin abrir cada .pkl a mano.

**¿Qué observas?** Con 3 "experimentos" ya es imposible responder con certeza qué AUC tuvo cada uno o con qué `C` se entrenó el segundo. Con 10 configuraciones (lo que compararemos en esta clase) un cuaderno así es directamente inmanejable: no se puede filtrar, ordenar ni comparar sin abrir cada archivo a mano. Eso es exactamente lo que resuelve MLflow Tracking.

---
## 2. Carpeta de trabajo efímera y almacén local de MLflow

Como en la clase 3: creamos `proyecto_churn/` desde cero, y dentro de ella (nunca en `datos/` ni en la carpeta del curso) el almacén de archivos de MLflow.

In [ ]:
CARPETA = "proyecto_churn"

# Limpiamos y recreamos la carpeta para que el notebook sea repetible desde cero
if os.path.exists(CARPETA):
    shutil.rmtree(CARPETA)
os.makedirs(CARPETA)
print("Carpeta de trabajo creada:", os.path.abspath(CARPETA))

In [ ]:
# El almacén de MLflow es una carpeta LOCAL dentro de la carpeta efímera: nunca un servidor
RUTA_MLRUNS = os.path.join(CARPETA, "mlruns")
mlflow.set_tracking_uri("file:" + RUTA_MLRUNS)

EXPERIMENTO = "churn_experimentos"
mlflow.set_experiment(EXPERIMENTO)

print("Tracking URI configurado:", mlflow.get_tracking_uri())
print("Experimento activo:", EXPERIMENTO)
assert mlflow.get_tracking_uri().startswith("file:") and "proyecto_churn" in mlflow.get_tracking_uri()

**¿Qué observas?** `mlflow.get_tracking_uri()` empieza por `file:` y contiene `proyecto_churn`: es un almacén de archivos local, dentro de la carpeta efímera de esta clase, no un servidor ni una carpeta compartida. Si borras `proyecto_churn/` (lo hacemos en la última celda), el historial de experimentos desaparece con ella: es exactamente lo que queremos en un notebook repetible.

---
## 3. Los datos y la línea base (por qué la exactitud sola engaña)

Cargamos `churn.csv`, dividimos en entrenamiento/prueba con una única semilla fija, y recordamos la proporción de abandono de `datos/LEEME.md`.

In [ ]:
RUTA_CSV = cargar_datos("churn.csv")
df = pd.read_csv(RUTA_CSV)

COLUMNAS_CATEGORICAS = ["plan", "contrato", "metodo_pago"]
COLUMNAS_NUMERICAS = ["antiguedad_meses", "uso_mensual_horas", "tickets_soporte", "factura_mensual"]

X = df[COLUMNAS_CATEGORICAS + COLUMNAS_NUMERICAS]
y = df["abandono"]

SEMILLA_BASE = 42
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEMILLA_BASE, stratify=y
)

proporcion_abandono = round(float(y.mean()), 4)
print("filas totales:", len(df))
print("proporción de abandono (dataset completo):", proporcion_abandono)
print("filas de entrenamiento / prueba:", len(X_train), "/", len(X_test))
assert len(X_train) + len(X_test) == len(df)

In [ ]:
def construir_preprocesador():
    """Mismo preprocesado para las 10 configuraciones: OneHotEncoder en categóricas, escalado en numéricas."""
    return ColumnTransformer([
        ("categoricas", OneHotEncoder(handle_unknown="ignore"), COLUMNAS_CATEGORICAS),
        ("numericas", StandardScaler(), COLUMNAS_NUMERICAS),
    ])


NOMBRE_HIPERPARAMETRO = {"regresion_logistica": "C", "arbol_decision": "profundidad_max"}


def entrenar_pipeline(modelo, class_weight, valor_hp, semilla, X_ent, y_ent):
    """Entrena una Pipeline completa (preprocesado + modelo) con una configuración concreta."""
    if modelo == "regresion_logistica":
        clasificador = LogisticRegression(class_weight=class_weight, random_state=semilla, max_iter=1000, C=valor_hp)
    else:
        clasificador = DecisionTreeClassifier(class_weight=class_weight, random_state=semilla, max_depth=valor_hp)
    pipeline = Pipeline([("preprocesado", construir_preprocesador()), ("clasificador", clasificador)])
    pipeline.fit(X_ent, y_ent)
    return pipeline, clasificador


def evaluar_pipeline(pipeline, clasificador, modelo, X_prueba, y_prueba):
    """Calcula las métricas y un indicador de coste computacional determinista (no el reloj de pared)."""
    predicciones = pipeline.predict(X_prueba)
    probabilidades = pipeline.predict_proba(X_prueba)[:, 1]
    exactitud = accuracy_score(y_prueba, predicciones)
    f1 = f1_score(y_prueba, predicciones, zero_division=0)
    auc = roc_auc_score(y_prueba, probabilidades)
    if modelo == "regresion_logistica":
        coste = int(clasificador.n_iter_[0])
    else:
        coste = int(clasificador.tree_.node_count)
    return predicciones, exactitud, f1, auc, coste

**¿Qué observas?** El `abandono` es del 11,7 % (coincide con `datos/LEEME.md`). Con una clase tan minoritaria, un modelo que **siempre** prediga "no abandona" ya acierta en torno al 88 % de las veces: la sección siguiente lo comprueba entrenando de verdad la configuración más simple.

---
## 4. Registrar un entrenamiento con MLflow, paso a paso

Entrenamos la configuración 1 (regresión logística, sin ajustar el desbalanceo) y la registramos con las cuatro llamadas clave de MLflow: `mlflow.start_run()`, `log_param`, `log_metric` y `log_artifact`. Como indicador de coste computacional usamos el **número de iteraciones** del solver (`n_iter_`): es determinista (no depende de la velocidad de tu máquina en este momento), a diferencia del tiempo de reloj, que también registramos (`tiempo_entrenamiento_seg`) pero no comparamos cifra a cifra en este notebook.

In [ ]:
# Config 1: regresión logística, sin ajustar el desbalanceo (la línea base "ingenua")
t0 = time.perf_counter()
pipeline_1, clasificador_1 = entrenar_pipeline("regresion_logistica", None, 1.0, SEMILLA_BASE, X_train, y_train)
tiempo_1 = time.perf_counter() - t0
predicciones_1, exactitud_1, f1_1, auc_1, coste_1 = evaluar_pipeline(pipeline_1, clasificador_1, "regresion_logistica", X_test, y_test)

with mlflow.start_run(run_name="config_1"):
    mlflow.set_tag("fase", "comparacion")
    # log_param: hiperparámetros y metadatos de ESTE entrenamiento concreto
    mlflow.log_param("config_id", 1)
    mlflow.log_param("modelo", "regresion_logistica")
    mlflow.log_param("class_weight", "ninguno")
    mlflow.log_param("hiperparametro", "C=1.0")
    mlflow.log_param("semilla", SEMILLA_BASE)
    # log_metric: resultados numéricos de ESTE entrenamiento concreto
    mlflow.log_metric("exactitud", exactitud_1)
    mlflow.log_metric("f1", f1_1)
    mlflow.log_metric("auc", auc_1)
    mlflow.log_metric("coste_computo", coste_1)
    mlflow.log_metric("tiempo_entrenamiento_seg", tiempo_1)
    # log_artifact: un archivo real generado por este entrenamiento (aquí, un informe de clasificación)
    informe_1 = classification_report(y_test, predicciones_1, zero_division=0)
    ruta_informe_1 = os.path.join(CARPETA, "informe_config_1.txt")
    with open(ruta_informe_1, "w") as f:
        f.write(informe_1)
    mlflow.log_artifact(ruta_informe_1)
    id_ejecucion_1 = mlflow.active_run().info.run_id

print("exactitud:", round(exactitud_1, 4))
print("f1:", round(f1_1, 4))
print("auc:", round(auc_1, 4))
print("coste_computo (iteraciones del solver):", coste_1)
assert round(exactitud_1, 3) == round(float((y_test == 0).mean()), 3), "la exactitud debería coincidir con la de predecir siempre 'no abandona'"

**¿Qué observas?** La exactitud (≈0,88) es prácticamente igual a la de predecir siempre "no abandona" (el `assert` lo comprueba), pero `f1` es 0,0: el modelo no identifica ni un solo cliente que realmente abandona. Mirando solo la exactitud, esta configuración parecería buena; mirando el f1, es inútil para el objetivo real (encontrar a quién va a abandonar).

---
## 5. Comparar 10 configuraciones reales

Definimos las 10 configuraciones (5 de regresión logística, 5 de árbol de decisión, variando `class_weight` y el hiperparámetro principal) y registramos las 9 restantes con la misma función.

In [ ]:
# Las 10 configuraciones que vamos a comparar: 5 de regresión logística, 5 de árbol de decisión,
# variando class_weight y el hiperparámetro principal de cada modelo
CONFIGURACIONES = [
    (1, "regresion_logistica", None, 1.0),
    (2, "regresion_logistica", None, 0.1),
    (3, "regresion_logistica", "balanced", 1.0),
    (4, "regresion_logistica", "balanced", 0.1),
    (5, "regresion_logistica", "balanced", 10.0),
    (6, "arbol_decision", None, 5),
    (7, "arbol_decision", None, 10),
    (8, "arbol_decision", "balanced", 5),
    (9, "arbol_decision", "balanced", 10),
    (10, "arbol_decision", "balanced", 3),
]


def registrar_configuracion(config_id, modelo, class_weight, valor_hp, semilla, X_ent, y_ent, X_prueba, y_prueba, fase="comparacion"):
    """Entrena, evalúa y registra en MLflow una configuración: una única función para las 10 + la estabilidad."""
    t0 = time.perf_counter()
    pipeline, clasificador = entrenar_pipeline(modelo, class_weight, valor_hp, semilla, X_ent, y_ent)
    tiempo = time.perf_counter() - t0
    predicciones, exactitud, f1, auc, coste = evaluar_pipeline(pipeline, clasificador, modelo, X_prueba, y_prueba)
    with mlflow.start_run(run_name=f"config_{config_id}_s{semilla}"):
        mlflow.set_tag("fase", fase)
        mlflow.log_param("config_id", config_id)
        mlflow.log_param("modelo", modelo)
        mlflow.log_param("class_weight", class_weight if class_weight else "ninguno")
        mlflow.log_param("hiperparametro", f"{NOMBRE_HIPERPARAMETRO[modelo]}={valor_hp}")
        mlflow.log_param("semilla", semilla)
        mlflow.log_metric("exactitud", exactitud)
        mlflow.log_metric("f1", f1)
        mlflow.log_metric("auc", auc)
        mlflow.log_metric("coste_computo", coste)
        mlflow.log_metric("tiempo_entrenamiento_seg", tiempo)
        informe = classification_report(y_prueba, predicciones, zero_division=0)
        ruta_informe = os.path.join(CARPETA, f"informe_config_{config_id}_s{semilla}.txt")
        with open(ruta_informe, "w") as f:
            f.write(informe)
        mlflow.log_artifact(ruta_informe)
        id_ejecucion = mlflow.active_run().info.run_id
    return pipeline, exactitud, f1, auc, coste, id_ejecucion

In [ ]:
# Ya registramos la config 1 a mano; ahora registramos las 9 restantes con la misma función
pipelines = {1: pipeline_1}
for config_id, modelo, class_weight, valor_hp in CONFIGURACIONES[1:]:
    pipeline, exactitud, f1, auc, coste, _ = registrar_configuracion(
        config_id, modelo, class_weight, valor_hp, SEMILLA_BASE, X_train, y_train, X_test, y_test
    )
    pipelines[config_id] = pipeline

print("configuraciones registradas en total:", len(pipelines))
assert len(pipelines) == 10

---
## 6. Leer los runs guardados con `mlflow.search_runs()`

`mlflow.search_runs()` devuelve un `DataFrame` de pandas con todos los runs del experimento. Seleccionamos solo las columnas `params.*`/`metrics.*` que registramos nosotros mismos.

In [ ]:
# mlflow.search_runs() devuelve un DataFrame de pandas con TODOS los runs del experimento activo
runs = mlflow.search_runs(experiment_names=[EXPERIMENTO])

# Nos quedamos SOLO con los parámetros y métricas que registramos nosotros mismos:
# nunca imprimimos start_time/end_time/run_id, que son metadatos automáticos de MLflow y no son reproducibles en texto
columnas_propias = [c for c in runs.columns if c.startswith("params.") or c.startswith("metrics.")]
columnas_propias = [c for c in columnas_propias if "tiempo_entrenamiento" not in c]

comparacion = runs[runs["tags.fase"] == "comparacion"][columnas_propias].copy()
comparacion["params.config_id"] = comparacion["params.config_id"].astype(int)
for columna in ["metrics.exactitud", "metrics.f1", "metrics.auc"]:
    comparacion[columna] = comparacion[columna].round(4)
comparacion = comparacion.sort_values("params.config_id").reset_index(drop=True)

print(comparacion[["params.config_id", "params.modelo", "params.class_weight", "params.hiperparametro",
                    "metrics.exactitud", "metrics.f1", "metrics.auc", "metrics.coste_computo"]].to_string(index=False))
assert len(comparacion) == 10

**¿Qué observas?** Los 5 árboles de decisión tienen un AUC claramente peor (0,56-0,67) que las 5 regresiones logísticas (0,72-0,72): aquí el modelo lineal generaliza mejor que el árbol, incluso con `class_weight="balanced"` en ambos. Dentro de las regresiones logísticas, las dos sin balancear (config. 1 y 2) tienen el AUC *más alto* de las diez y, sin embargo, f1 = 0: si el criterio fuera "AUC más alto y ya está", elegirías precisamente el modelo inútil de la sección 4.

---
## 7. Un criterio explícito para elegir

No basta con "la métrica principal más alta" (sección 6). Aplicamos tres filtros, en orden:

1. **AUC ≥ 0,70** (no depende del umbral): descarta los árboles de decisión.
2. De los que quedan, **F1 ≥ 0,30** (con umbral 0,5): descarta las dos regresiones sin balancear.
3. Entre los finalistas, **menor `coste_computo`** primero (menos iteraciones del solver, entrenamiento más barato).

In [ ]:
# Criterio explícito, en tres pasos, NO solo "el AUC más alto":
# 1) AUC >= 0.70 (descarta los árboles, que generalizan peor aquí)
# 2) de los que quedan, F1 >= 0.30 (descarta las dos regresiones sin balancear, con f1 = 0)
# 3) entre los finalistas, menor coste_computo primero (menos iteraciones del solver)
AUC_MINIMO = 0.70
F1_MINIMO = 0.30

registros = comparacion.to_dict("records")
supera_auc = [r for r in registros if r["metrics.auc"] >= AUC_MINIMO]
supera_f1 = [r for r in supera_auc if r["metrics.f1"] >= F1_MINIMO]
finalistas = sorted(supera_f1, key=lambda r: r["metrics.coste_computo"])

print("configuraciones que superan AUC >=", AUC_MINIMO, ":", [r["params.config_id"] for r in supera_auc])
print("...y además F1 >=", F1_MINIMO, ":", [r["params.config_id"] for r in supera_f1])
print("finalistas ordenados por coste (menor primero):", [(r["params.config_id"], r["metrics.coste_computo"]) for r in finalistas])

assert {r["params.config_id"] for r in supera_f1} == {r["params.config_id"] for r in finalistas}
assert [r["params.config_id"] for r in supera_auc] == [1, 2, 3, 4, 5]
assert [r["params.config_id"] for r in supera_f1] == [3, 4, 5]

**¿Qué observas?** Tres configuraciones (3, 4 y 5) superan los dos filtros: las tres son regresión logística con `class_weight="balanced"`, y solo cambian en `C`. El AUC (0,7227-0,7232) y el F1 (0,3085-0,3166) son casi idénticos entre ellas; el desempate por coste ya da una primera preferencia (config. 3, con menos iteraciones), pero antes de decidir del todo comprobamos si ese resultado es estable entre semillas.

---
## 8. Estabilidad entre semillas

Una sola semilla (42) puede haber sido "afortunada" para alguna de las tres finalistas. Repetimos las tres con dos semillas más (7 y 123, fijas para que el resultado sea reproducible) y miramos la media y la desviación de F1 y AUC.

In [ ]:
# Mirar solo la semilla 42 puede engañar: repetimos los 3 finalistas con dos semillas más
SEMILLAS_EXTRA = [7, 123]
IDS_FINALISTAS = [r["params.config_id"] for r in finalistas]
CONFIG_POR_ID = {c[0]: c for c in CONFIGURACIONES}

f1_por_config = {cid: [next(r["metrics.f1"] for r in registros if r["params.config_id"] == cid)] for cid in IDS_FINALISTAS}
auc_por_config = {cid: [next(r["metrics.auc"] for r in registros if r["params.config_id"] == cid)] for cid in IDS_FINALISTAS}

for semilla in SEMILLAS_EXTRA:
    X_ent_s, X_prueba_s, y_ent_s, y_prueba_s = train_test_split(X, y, test_size=0.25, random_state=semilla, stratify=y)
    for cid in IDS_FINALISTAS:
        _, modelo, class_weight, valor_hp = CONFIG_POR_ID[cid]
        _, exactitud, f1, auc, coste, _ = registrar_configuracion(
            cid, modelo, class_weight, valor_hp, semilla, X_ent_s, y_ent_s, X_prueba_s, y_prueba_s, fase="estabilidad"
        )
        f1_por_config[cid].append(f1)
        auc_por_config[cid].append(auc)

estabilidad = []
for cid in IDS_FINALISTAS:
    f1s = np.array(f1_por_config[cid])
    aucs = np.array(auc_por_config[cid])
    estabilidad.append((cid, round(f1s.mean(), 4), round(f1s.std(), 4), round(aucs.mean(), 4), round(aucs.std(), 4)))
    print(f"config {cid}: f1 media={round(f1s.mean(),4)} f1 desviacion={round(f1s.std(),4)}  auc media={round(aucs.mean(),4)} auc desviacion={round(aucs.std(),4)}")

# Verificamos que las ejecuciones de estabilidad quedaron guardadas de verdad en MLflow
runs_estabilidad = mlflow.search_runs(experiment_names=[EXPERIMENTO], filter_string="tags.fase = \'estabilidad\'")
print("ejecuciones de estabilidad guardadas en MLflow:", len(runs_estabilidad))
assert len(runs_estabilidad) == len(IDS_FINALISTAS) * len(SEMILLAS_EXTRA)

**¿Qué observas?** Con tres semillas, el F1 medio de las tres configuraciones sigue siendo prácticamente el mismo (≈0,31), pero la **desviación** ya las distingue: la config. 3 (`C=1.0`) es la más estable (desviación de F1 más baja). Sin este paso, habrías elegido solo por la primera semilla, sin saber si el resultado se sostiene.

---
## 9. Registrar el modelo elegido como artefacto versionado

Entrenamos de nuevo la configuración ganadora (semilla base) y la registramos con `mlflow.sklearn.log_model`: el modelo entrenado queda guardado como un artefacto más de MLflow, versionado junto a sus parámetros y métricas. Esto no es todavía el registro de modelos de la clase 8 (no hay versiones con nombre ni *stages*), pero es la pieza que ese registro necesita para existir.

In [ ]:
# El ganador: menor desviación de F1 entre semillas (más estable) Y menor coste, con AUC y F1 ya validados
ganador_id, f1_media, f1_std, auc_media, auc_std = min(estabilidad, key=lambda t: t[2])
_, modelo_ganador, class_weight_ganador, hp_ganador = CONFIG_POR_ID[ganador_id]

print("configuración elegida:", ganador_id, modelo_ganador, class_weight_ganador, hp_ganador)
print("f1 media entre semillas:", f1_media, "(desviacion:", f1_std, ")")

pipeline_ganador, clasificador_ganador = entrenar_pipeline(modelo_ganador, class_weight_ganador, hp_ganador, SEMILLA_BASE, X_train, y_train)
predicciones_ganador, exactitud_ganador, f1_ganador, auc_ganador, coste_ganador = evaluar_pipeline(
    pipeline_ganador, clasificador_ganador, modelo_ganador, X_test, y_test
)

with mlflow.start_run(run_name="modelo_elegido"):
    mlflow.set_tag("fase", "modelo_elegido")
    mlflow.log_param("config_id", ganador_id)
    mlflow.log_param("modelo", modelo_ganador)
    mlflow.log_param("class_weight", class_weight_ganador)
    mlflow.log_param("hiperparametro", f"{NOMBRE_HIPERPARAMETRO[modelo_ganador]}={hp_ganador}")
    mlflow.log_param("semilla", SEMILLA_BASE)
    mlflow.log_metric("exactitud", exactitud_ganador)
    mlflow.log_metric("f1", f1_ganador)
    mlflow.log_metric("auc", auc_ganador)
    resumen = (
        f"Configuracion elegida: {ganador_id} ({modelo_ganador}, class_weight={class_weight_ganador})\n"
        f"Criterio: AUC >= {AUC_MINIMO}, F1 >= {F1_MINIMO}, menor coste_computo, menor desviacion de F1 entre semillas.\n"
        f"f1 media entre semillas: {f1_media} (desviacion {f1_std})\n"
    )
    ruta_resumen = os.path.join(CARPETA, "resumen_modelo_elegido.txt")
    with open(ruta_resumen, "w") as f:
        f.write(resumen)
    mlflow.log_artifact(ruta_resumen)
    # log_model: registra el propio modelo entrenado como artefacto versionado (paso previo al registro de modelos de la clase 8)
    mlflow.sklearn.log_model(pipeline_ganador, name="modelo", pip_requirements=[f"scikit-learn=={sklearn.__version__}"])
    id_ejecucion_ganadora = mlflow.active_run().info.run_id

print("modelo guardado como artefacto de MLflow (dentro de la carpeta efímera).")

In [ ]:
# Verificación concreta: recargamos el modelo desde MLflow y comprobamos que predice IGUAL
modelo_recargado = mlflow.sklearn.load_model(f"runs:/{id_ejecucion_ganadora}/modelo")
predicciones_recargadas = modelo_recargado.predict(X_test)

print("¿las predicciones del modelo recargado coinciden con las originales?", bool((predicciones_recargadas == predicciones_ganador).all()))
assert (predicciones_recargadas == predicciones_ganador).all()

**¿Qué observas?** El modelo recargado desde el almacén de MLflow (`mlflow.sklearn.load_model`) predice exactamente igual que el original: el artefacto guardado es una copia fiel y funcional del modelo entrenado, no solo un registro de sus métricas.

---
## 10. Limpieza final

Eliminamos `proyecto_churn/` (con su almacén de MLflow y todos los artefactos) para que no quede ningún residuo dentro del repositorio del curso, ni siquiera al reejecutar este notebook.

In [ ]:
# Limpieza final: nada de esta clase debe quedar dentro del repositorio del curso
shutil.rmtree(CARPETA)
print("Carpeta de trabajo eliminada:", CARPETA)
assert not os.path.exists(CARPETA)

---
## Resumen de la clase

- Anotar experimentos "a mano" deja de funcionar en cuanto hay más de un puñado: no se puede filtrar ni comparar sin abrir cada archivo.
- **MLflow Tracking**, con un almacén de archivos **local** (nunca un servidor en este curso), registra parámetros, métricas y artefactos de cada entrenamiento real con `start_run`/`log_param`/`log_metric`/`log_artifact`.
- Comparamos **10 configuraciones reales** (regresión logística y árbol de decisión, con y sin `class_weight="balanced"`): el AUC más alto (regresión sin balancear) correspondía al modelo con f1 = 0, la trampa exacta de mirar una sola métrica.
- Un **criterio explícito** (umbral de AUC, umbral de F1, luego coste) y la **estabilidad entre semillas** son necesarios para no elegir por casualidad.
- `mlflow.sklearn.log_model` registra el modelo elegido como **artefacto versionado**: el primer paso hacia el registro de modelos de la clase 8.

*Continúa con la clase 5: Pipelines reproducibles.*